<hr>

```text
                                 _         __      _       _                 _    
  /\  /\__ _ _ ____   _____  ___| |_    /\ \ \___ | |_ ___| |__   ___   ___ | | __
 / /_/ / _` | '__\ \ / / _ \/ __| __|  /  \/ / _ \| __/ _ \ '_ \ / _ \ / _ \| |/ /
/ __  / (_| | |   \ V /  __/\__ \ |_  / /\  / (_) | ||  __/ |_) | (_) | (_) |   < 
\/ /_/ \__,_|_|    \_/ \___||___/\__| \_\ \/ \___/ \__\___|_.__/ \___/ \___/|_|\_\
```

<br>
<hr>

## Program Overview

Program: **Harvest Notebook**<br>
Developed by: **David Blessent**<br>
Publisher: **Almond House Publishing**<br>
Repository: **https://github.com/almondhouse27/harvest-notebook**<br>

#### Description 

A modular and interactive web scraper designed to be flexible, easy-to-use, and reliable for generating datasets.<br>
Harvest Notebook allows users to input a list of URLs, check robots.txt files for permissions, scrape data from websites, and generate a suite of comprehensive reports.<br>
Reports include page specifications, extracted data, and diagnostics of the scraping process, with optional features for additional processing and analysis. <br>
Built with a Jupyter Notebook interface for user interactivity and Python modules for robust core functionality.

#### Contact

If you have any questions, comments, concerns, or just want to talk data, feel free to reach out to me at: `almondhousepublishing@protonmail.com` <br>
I very likely **will be slow** to respond, but *I will get back to you*. Add "Harvest Notebook" to the subject to help me out!<br>
Submit any issues with and suggestions for the Notebook, its code, and its features through the GitHub repository listed above.<br>
I welcome constructive, respectful feedback and am happy to assist with any inquiries!

<hr>

## How To Use Harvest Notebook

<hr>

## Notebook Setup

#### Imports

In [ ]:
"""
imports python libraries
    os (operating system interface)
        os.path.join() for creating file paths in a platform-independent way
    re (regular expressions)
        re.compile() to precompile regular expression for url validation

imports Harvest Notebook modules from: ./src/

moonlights as a function directory
"""

import os
import re

from src.config     import install_requirements, setup_logging
from src.input      import read_input, input_review
from src.cleaner    import clean_data
from src.scraper    import harvest
from src.diagnostic import process_diagnostic_log
from src.output     import create_output_directory, stitch_chunks, \
                           basic_report_processing, clean_log

# helper function for easy-to-provide user feedback
def complete(task):
    print(f"{task} complete!")

complete("Standard library and custom module imports")

#### Assign Paths To Constants

In [ ]:
"""
constants provide filepaths for input, output, and logging
    enables user to adjust filepaths according to preference

constructs filepaths using os.path.join() for platform independence
"""

LOG = os.path.join('data', 'log', 'app.log')
ARCHIVE = os.path.join('data', 'log', 'archive.log')
INPUT = os.path.join('data', 'input', 'data.csv')
BAD_DATA = os.path.join('data', 'input', 'bad_data.csv')
OUTPUT = os.path.join('data', 'output')
WORD = "website_words.csv"
SPEC = "site_specifications.csv"

complete("Filepath assignments")

#### Input Validation

In [ ]:
"""
constants for input data column validation and url regex
    enables user to adjust shape of input data
    enables user to adjust url validation regex (not recommended unless proficient in regex)
"""

COLUMNS = [ 'Category', 'Name', 'Url' ]
URL_CHECK = re.compile(r'^(https?://)?([a-z0-9]+([-\w]*[a-z0-9])*\.)+[a-z]{2,6}(:[0-9]{1,5})?(/.*)?$', re.IGNORECASE)
complete("Validation assignments")

#### Optional Features

In [ ]:
"""
constants enable user to turn optional features on/off
    functions are executed in "Process Output Files" section 

constants control methods contained in files contained in: ./src/option/
    RUN_ARP     advanced.py     advanced_report_processing()
    RUN_HVS:    visuals.py      harvest_visualization_suite()
    RUN_PIA:    patterns.py     pattern_identitification_analysis()
"""

RUN_ARP = True
RUN_HVS = False
RUN_PIA = False
complete("Setting optional features")

#### Configuration

In [ ]:
"""
runs function(s) contained in: ./src/config.py

setup_logging(LOG)
    log output accumulates in: ./data/log/app.log

install_requirements()
    calls pip install requrirements.txt with subprocess library 
        python libraries: [pandas, requests] 
"""

setup_logging(LOG)
install_requirements()
complete("Notebook configuration")

<hr>

## Prepare The DataFrame

#### Read Data File

In [ ]:
"""
runs function(s) contained in: ./src/input_data.py

read_input(INPUT, COLUMNS, BAD_DATA)
    reads list of URLs from: ./data/input/data.csv 
    checks that formatting of csv data matches column requirements: ['Category', 'Name', 'Url']
        if not matches do: 
            - copies ./data/input/data.csv to ./data/input/bad_data.csv 
            - clears ./data/input/data.csv and writes correct column headers
        if matches do:
            - returns: df (dataframe)
"""

df = read_input(INPUT, COLUMNS, BAD_DATA)
complete("Input data read")

#### Summarize Data

In [ ]:
"""
runs function(s) contained in: ./src/cleaner.py

input_review(df) 
    provides an overview of the dataframe's structure and content
"""

input_review(df)
complete("Data exploration")

#### Clean Data

In [ ]:
"""
runs function(s) contained in:./src/cleaner.py

clean_data(df, URL_CHECK, KEEP) calls:
    handle_missing_data(df, URL_CHECK) 
        summarizes missing data in the dataframe
        filters out rows with invalid/missing Url values using URL_CHECK regex
        replaces missing Category and Name values with "Unavailable"

    handle_duplicate_urls(df, keep)
        removed duplicate URLs from the dataframe
        keeps the first instance of the URL
            adjustable 'keep' parameter
"""

KEEP = 'first'
df, clean_summary = clean_data(df, URL_CHECK, KEEP)
print(clean_summary)
complete("Handle missing values")

<hr>

## Run The Scraper

#### Create Timestamped Directory

In [ ]:
"""
runs function(s) contained in: ./src/output.py

create_output_directory(OUTPUT)
    checks if the specified OUTPUT directory exists, creates it if not
    creates a unique timestamped subdirectory inside OUTPUT
    if a directory with timestamp already exists, it appends an incrementing counter to filename
"""

output_dir = create_output_directory(OUTPUT)
print(f"Output path: {output_dir}")
complete("Output directory creation")

#### Visit Urls

In [ ]:
"""
runs function(s) contained in: ./src/scraper.py

harvest(df, output_dir)
    this function iterates over a DataFrame containing website URLs
    checks if the URL can be accessed based on robots.txt restrictions
    ends a GET request to the URL with random user-agent and proxy settings
    extracts word counts and site specifications from the HTML content
    saves the collected data in chunks to the output directory
    logs processing information to app.log
"""

harvest(df, output_dir)
complete("Harvest")

<hr>

## Process Output Files

#### Assemble Raw Data

In [ ]:
"""
runs function(s) contained in: ./src/output.py

stitch_chunks(output_dir, WW, SS)
    defines the paths for the final output files
    collects and sorts the chunked CSV files for website words and site specifications
    concatenates the chunked CSV files into single DataFrames
    saves the concatenated DataFrames as final CSV files
    removes the raw chunk directory and its files
"""

stitch_chunks(output_dir, WORD, SPEC)
complete("Raw chunk stitching")

#### Basic Report Processing

In [ ]:
"""
runs function(s) contained in: ./src/output.py

basic_report_processing(output_dir)
    collects all CSV files in the output directory
    for each CSV file:
        reads the file into a DataFrame
        strips whitespace from string values
        sorts the DataFrame by the first one or two columns
        saves the cleaned and sorted DataFrame back to the original file
"""

basic_report_processing(output_dir)
complete("Basic report processing")

#### Instance Records

In [ ]:
"""
runs function(s) contained in: ./src/output.py & ./src/diagnostic.py

process_diagnostic_log(LOG, ARCHIVE, output_dir)
    reads the log file app.log
    extracts and summarizes key information
    saves the summarized output to diagnotic_log.json
    copies app.log to archive.log and clears app.log
"""

clean_log(clean_summary, output_dir)
process_diagnostic_log(LOG, ARCHIVE, output_dir)
complete("")

<hr>

## Optional Features

#### Advanced Report Processing

**Notebook Setup > Optional Features**<br>set to activate Advanced Report Processing:<br>
`RUN_ARP = True`

In [ ]:
"""
runs function(s) contained in: ./src/option/advanced.py

advanced_report_processing(output_dir)
    performs additional processing on the website_words.csv and site_specifications.csv output files
        website_words:
            tokenizes 'Word' column, removes stop words, adds sentiment columns according to
        site_specifications:
            replaces missing values with "Unavailable"
"""

if RUN_ARP:
    from src.option.advanced import advanced_report_processing
    advanced_report_processing(output_dir, WORD, SPEC)
    complete("Advanced report processing")

#### Harvest Visualization Suite

**Notebook Setup > Optional Features**<br>
set to activate Harvest Visualization Suite<br>
`RUN_HVS = True`

In [15]:
""" 
runs function(s) contained in: ./src/option/visuals.py

harvest_visualization_suite(output_dir)
    generates built-in visualizations for website_words.csv and site_specifications.csv output files
        saves image files to: ./data/output/{output_dir}/visuals/
"""

if RUN_HVS:
    from src.option.visuals import harvest_visualization_suite
    harvest_visualization_suite(output_dir)
    complete("Harvest visualization suite")

#### Pattern Identification Analysis

**Notebook Setup > Optional Features**<br>
set to activate Pattern Identification Analysis<br>
`RUN_PIA = True`

In [16]:
"""
runs function(s) contained in: ./src/option/pattern.py

pattern_identification_analysis(output_dir)
    generates new output report
"""

if RUN_PIA:
    from src.option.patterns import pattern_identification_analysis
    pattern_identification_analysis(output_dir)
    complete("Pattern identification analysis")

<hr>